# Welcome to HiMaLAYAS Quickstart

This notebook provides a minimal working example of **HiMaLAYAS (Hierarchical Matrix Layout and Annotation Software)** for post hoc enrichment-based annotation of hierarchically clustered matrices.

In this quickstart, we apply HiMaLAYAS to a **yeast genetic interaction profile similarity matrix** (Costanzo et al., 2016) and annotate clusters with **GO Biological Process** terms. The matrix focuses on genes with high profile variance (~1,100 genes in the publication workflow).

You will learn how to:
- Load a matrix and annotations
- Cluster and enrich the matrix
- Filter by q-value and summarize clusters
- Plot an annotated, publication-ready figure

Expected input files in `data/yeast`:
- `gi_pcc_sampled.tsv`
- `go_bp_term_to_orfs.json`
- `go_id_to_name.json`


In [ ]:
# Imports and setup
import os
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import himalayas as h
from himalayas import Matrix, Annotations, Analysis
from himalayas.text import summarize_clusters
from himalayas.plot import Plotter

print(f"HiMaLAYAS version: {h.__version__}")

# Set working directory if running in a notebook environment
if "__file__" not in globals():
    os.chdir(Path().resolve())

# Enable inline plotting for notebooks
%matplotlib inline

---

## Load GO Biological Process Annotations

We load term-to-gene mappings and a GO ID to name lookup for presentation.


In [ ]:
DATA_DIR = Path("data/yeast")
GO_BP_PATH = DATA_DIR / "go_bp_term_to_orfs.json"
GO_ID_TO_NAME_PATH = DATA_DIR / "go_id_to_name.json"

with GO_BP_PATH.open("r", encoding="utf-8") as fh:
    go_bp = json.load(fh)

with GO_ID_TO_NAME_PATH.open("r", encoding="utf-8") as fh:
    go_id_to_name = json.load(fh)

term_sizes = [len(set(orfs)) for orfs in go_bp.values()]
all_orfs = {orf for orfs in go_bp.values() for orf in orfs}

print(f"GO BP terms loaded: {len(term_sizes):,}")
print(f"Min term size: {min(term_sizes)}")
print(f"Max term size: {max(term_sizes)}")
print(f"Unique ORFs across all terms: {len(all_orfs):,}")

---

## Load the Matrix

We load a gene-by-gene similarity matrix (PCC). Rows and columns should have identical labels.


In [ ]:
MATRIX_PATH = DATA_DIR / "gi_pcc_sampled.tsv"

DF = pd.read_csv(
    MATRIX_PATH,
    sep="	",
    index_col=0,
)

print(f"Matrix shape: {DF.shape[0]:,} x {DF.shape[1]:,}")
print(f"Row/column labels identical: {DF.index.equals(DF.columns)}")
print(f"Value range: [{DF.min().min():.3f}, {DF.max().max():.3f}]")

DF.head()

---

## Cluster and Enrich

We build the core objects, run hierarchical clustering, perform hypergeometric enrichment, and compute q-values.


In [ ]:
LINKAGE_THRESHOLD = 16

matrix = Matrix(DF)
annotations = Annotations(go_bp, matrix)

analysis = (
    Analysis(matrix, annotations)
    .cluster(
        linkage_method="ward",
        linkage_metric="euclidean",
        linkage_threshold=LINKAGE_THRESHOLD,
        min_cluster_size=30,
    )
    .enrich(min_overlap=2)
    .finalize(col_cluster=True, add_qvalues=True)
)

results = analysis.results

# Map GO term IDs to human-readable names (presentation only)
results.df["term_name"] = results.df["term"].map(go_id_to_name).fillna(results.df["term"])

# Keep significant terms
results_sig = results.filter("qval <= 0.05")

# Summarize clusters for plotting
cluster_labels = summarize_clusters(
    results_sig.df,
    label_mode="top_term",
    term_col="term_name",
)

print(f"All enriched rows: {len(results.df):,}")
print(f"Significant rows (q<=0.05): {len(results_sig.df):,}")
print(cluster_labels.head())

---

## Plot the Annotated Matrix

We render a publication-ready figure using the `Plotter`.


In [ ]:
from matplotlib.colors import Normalize

LABEL_COLOR = "black"
BACKGROUND_COLOR = "white"

vals = matrix.values
mask = np.isfinite(vals) & (vals != 0)
vlim = float(np.percentile(np.abs(vals[mask]), 99))

plotter = (
    Plotter(results)
    .set_background(color=BACKGROUND_COLOR)
    .plot_title(
        "HiMaLAYAS - Yeast Genetic Interaction Similarity Matrix",
        color=LABEL_COLOR,
        fontsize=17,
    )
    .plot_dendrogram(
        axes=[0.06, 0.16, 0.09, 0.79],
        data_pad=0.35,
        color="#888888",
        linewidth=0.75,
    )
    .plot_matrix(
        cmap="RdBu_r",
        center=0,
        vmin=-vlim,
        vmax=vlim,
        outer_lw=0,
        figsize=(14, 10),
        subplots_adjust={"left": 0.15, "right": 0.62, "bottom": 0.16, "top": 0.95},
    )
    .plot_matrix_axis_labels(
        xlabel="Gene",
        ylabel="Gene",
        fontsize=16,
        font="Helvetica",
        color=LABEL_COLOR,
        xlabel_pad=6.0,
        ylabel_pad=0.007,
    )
    .plot_cluster_labels(
        cluster_labels,
        max_words=24,
        wrap_text=True,
        wrap_width=40,
        overflow="wrap",
        font="Helvetica",
        fontsize=17,
        color=LABEL_COLOR,
        skip_unlabeled=False,
        label_fields=("label", "p"),
        omit_words={},
        boundary_color=LABEL_COLOR,
        boundary_lw=1,
        boundary_alpha=0.8,
        dendro_boundary_alpha=0.0,
        label_text_pad=0.012,
        label_sep_xmin=None,
        label_sep_xmax=0.5,
        label_sep_color=LABEL_COLOR,
        label_sep_lw=1,
        label_sep_alpha=0.4,
        label_gutter_color=BACKGROUND_COLOR,
        axes=[0.62, 0.16, 0.36, 0.79],
    )
    .plot_cluster_bar(
        norm=Normalize(0, 30),
        name="sigbar",
        title="Enrichment",
        values=cluster_labels,
        pval_col="pval",
        width=0.04,
        left_pad=0.06,
        right_pad=0.01,
    )
    .add_colorbar(
        name="matrix",
        cmap="RdBu_r",
        norm=Normalize(-vlim, vlim),
        label="PCC similarity",
        ticks=[-vlim, 0, vlim],
    )
    .add_colorbar(
        name="enrichment",
        cmap="YlOrBr",
        norm=Normalize(0, 30),
        label="-log10(p) enrichment",
        ticks=[0, 10, 20, 30],
    )
    .plot_colorbars(
        ncols=2,
        height=0.045,
        gap=0.05,
        hpad=0.04,
        vpad=0.00,
        fontsize=14,
        font="Helvetica",
        color=LABEL_COLOR,
        border_color=LABEL_COLOR,
        border_width=1.0,
        border_alpha=0.9,
    )
)

plotter.show()

---

## Optional: Zoom into a Single Cluster

Use `Results.subset()` to re-run clustering and enrichment for a single cluster.


In [ ]:
def run_zoom_analysis(
    *,
    results,
    cluster_id,
    go_bp,
    go_id_to_name,
    linkage_threshold,
    min_cluster_size=6,
    min_overlap=2,
    qval_cutoff=0.05,
):
    zoom_view = results.subset(cluster=cluster_id)
    zoom_matrix = zoom_view.matrix

    zoom_annotations = Annotations(go_bp, zoom_matrix)

    zoom_analysis = (
        Analysis(zoom_matrix, zoom_annotations)
        .cluster(
            linkage_method="ward",
            linkage_metric="euclidean",
            linkage_threshold=linkage_threshold,
            min_cluster_size=min_cluster_size,
        )
        .enrich(min_overlap=min_overlap, background=results.matrix)
        .finalize(col_cluster=True, add_qvalues=True)
    )

    zoom_results = zoom_analysis.results
    zoom_results.df["term_name"] = (
        zoom_results.df["term"].map(go_id_to_name).fillna(zoom_results.df["term"])
    )

    zoom_results_sig = zoom_results.filter(f"qval <= {qval_cutoff}")
    zoom_cluster_labels = summarize_clusters(
        zoom_results_sig.df,
        label_mode="top_term",
        term_col="term_name",
    )
    return zoom_matrix, zoom_results, zoom_results_sig, zoom_cluster_labels


# Example usage:
# zoom_matrix, zoom_results, zoom_results_sig, zoom_cluster_labels = run_zoom_analysis(
#     results=results,
#     cluster_id=7,
#     go_bp=go_bp,
#     go_id_to_name=go_id_to_name,
#     linkage_threshold=6,
# )

---

## Next Steps

- See the full documentation for parameter details and plotting options.
- Explore the example notebooks for additional workflows.
